# 🤖 ASSIGNMENT 13 — Agentic AI & Hệ đa tác tử
## Research Agent: tự tìm kiếm, đọc, viết báo cáo có nguồn (LangGraph)

> **Trọng tâm:** Kiến trúc agent + **guardrail**. Agent nhận chủ đề → tự nghiên cứu nhiều bước →
> xuất báo cáo có trích dẫn. Có tools, memory, observability, và giới hạn bước chống loop vô hạn.

> ⚠️ **Môi trường:** Kiến trúc agent (vòng lặp LangGraph + tools + memory + guardrail) **chạy thật**
> trong notebook này, dùng **tool giả lập + router rule-based** thay cho LLM (vì sandbox không tải được
> LLM weights). Cơ chế agent giữ nguyên — lên Colab chỉ cần thay node bằng LLM thật + web_search thật.

In [1]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
import operator
print("LangGraph sẵn sàng")

LangGraph sẵn sàng


## Câu 1 — Kiến trúc agent

```
                    ┌─────────────────────────────────┐
                    │           AGENT LOOP            │
                    │                                 │
   [Chủ đề] ──────► │  ┌──────┐   router    ┌──────┐ │
                    │  │ LLM  │─────────────►│Tools │ │
                    │  │(não) │◄─────────────│      │ │
                    │  └──────┘   kết quả    └──────┘ │
                    │      ▲                    │      │
                    │      │   ┌──────────┐    │      │
                    │      └───│  MEMORY  │◄───┘      │
                    │          │(findings)│           │
                    │          └──────────┘           │
                    │      │ guardrail: max_steps     │
                    └──────┼──────────────────────────┘
                           ▼
                    [Báo cáo có trích dẫn]
```

**4 thành phần:**
- **LLM (bộ não):** quyết định bước tiếp theo — tìm thêm hay viết báo cáo.
- **Tools:** web_search, fetch_page, write_section — tay chân để tương tác thế giới.
- **Memory:** tích lũy findings qua các bước (state) để agent "nhớ" đã tìm được gì.
- **Vòng lặp + guardrail:** lặp tới khi đủ thông tin HOẶC chạm giới hạn bước (chống loop vô hạn).

## Câu 2 — Pattern: ReAct vs Plan-Execute

**Chọn ReAct (Reasoning + Acting)** cho bài này.

| Pattern | Cách hoạt động | Hợp khi |
|---------|----------------|---------|
| **ReAct** | Lặp: Suy nghĩ → Hành động → Quan sát → lặp lại | Nhiệm vụ khám phá, không biết trước cần mấy bước |
| Plan-Execute | Lập kế hoạch đầy đủ trước → thực thi từng bước | Nhiệm vụ rõ ràng, các bước cố định |

**Vì sao ReAct hợp research agent?** Nghiên cứu là **khám phá** — không biết trước cần tìm bao nhiêu nguồn,
nội dung tìm được sẽ định hướng bước kế. ReAct cho agent **linh hoạt**: tìm được nguồn tốt thì dừng sớm,
nguồn yếu thì tìm thêm. Plan-Execute cứng nhắc hơn, hợp quy trình cố định (vd điền form theo bước).

## Câu 3 & 4 — Định nghĩa tools (mô tả rõ ràng)

In [2]:
# State = memory giữa các bước
class AgentState(TypedDict):
    topic: str
    findings: Annotated[List[str], operator.add]   # operator.add -> TÍCH LŨY (không ghi đè)
    step: int
    max_steps: int
    report: str

# ===== TOOLS (giả lập; production thay bằng web_search + fetch thật) =====
def web_search(topic: str) -> str:
    '''Tìm kiếm web về chủ đề, trả về đoạn tóm tắt kèm nguồn.
    Tham số: topic (str) - chủ đề cần tìm. Trả về: str chứa thông tin + [Nguồn].'''
    return f"[Nguồn 1] {topic}: lĩnh vực đang tăng trưởng mạnh, nhiều ngân hàng VN đã triển khai."

def fetch_page(topic: str) -> str:
    '''Đọc chi tiết một trang/tài liệu về chủ đề.
    Tham số: topic (str). Trả về: str nội dung chi tiết + [Nguồn].'''
    return f"[Nguồn 2] Báo cáo 2025: 60% ngân hàng VN dùng AI cho CSKH và phát hiện gian lận."

def write_section(topic: str, findings: List[str]) -> str:
    '''Tổng hợp findings thành báo cáo có trích dẫn.
    Tham số: topic (str), findings (list). Trả về: báo cáo markdown.'''
    body = "\n".join(findings)
    return f"# Báo cáo: {topic}\n\n{body}\n\n*(Tổng hợp từ {len(findings)} nguồn)*"

print("Đã định nghĩa 3 tools: web_search, fetch_page, write_section")

Đã định nghĩa 3 tools: web_search, fetch_page, write_section


**Vì sao mô tả tool tốt lại CỰC quan trọng?**
LLM **chọn tool dựa hoàn toàn vào phần mô tả (docstring)** — nó không thấy code bên trong. Mô tả mơ hồ →
agent gọi sai tool, sai tham số, hoặc không biết khi nào dùng. Mô tả tốt cần: **tên rõ nghĩa**, **chức năng
một câu**, **tham số + kiểu**, **giá trị trả về**. Đây là "API contract" giữa LLM và tool — viết kém là
nguyên nhân hàng đầu khiến agent hành xử thất thường.

## Câu 5 — Memory giữa các bước

In [3]:
# Memory được cài qua State với operator.add -> findings TÍCH LŨY qua mỗi bước
# Mỗi node trả về {"findings": [điều mới]} -> LangGraph tự GỘP vào list cũ

def search_node(state: AgentState):
    print(f"  [Bước {state['step']+1}] 🔍 web_search")
    return {"findings": [web_search(state["topic"])], "step": state["step"]+1}

def fetch_node(state: AgentState):
    print(f"  [Bước {state['step']+1}] 📄 fetch_page")
    return {"findings": [fetch_page(state["topic"])], "step": state["step"]+1}

def write_node(state: AgentState):
    print(f"  [Bước {state['step']+1}] ✍️ write_section")
    return {"report": write_section(state["topic"], state["findings"]),
            "step": state["step"]+1}
print("Nodes + memory (tích lũy findings) đã định nghĩa")

Nodes + memory (tích lũy findings) đã định nghĩa


**Cách memory hoạt động:** `Annotated[List[str], operator.add]` báo LangGraph: khi node trả về `findings`,
hãy **CỘNG DỒN** vào list cũ thay vì ghi đè. Nhờ vậy agent "nhớ" mọi phát hiện qua các bước → bước write
có đủ toàn bộ nguồn đã thu thập. Đây là **short-term memory** trong 1 lần chạy.

## Câu 6 — Vòng lặp agent (LangGraph) — chạy thật ✅

In [4]:
# Router = bộ não quyết định (production: LLM; ở đây: rule-based để chạy thật)
def router(state: AgentState):
    if state["step"] >= state["max_steps"]:    # GUARDRAIL
        return "write"
    if len(state["findings"]) < 2:             # chưa đủ nguồn -> tìm tiếp
        return "fetch"
    return "write"                             # đủ -> viết báo cáo

# Build graph
g = StateGraph(AgentState)
g.add_node("search", search_node)
g.add_node("fetch", fetch_node)
g.add_node("write", write_node)
g.set_entry_point("search")
g.add_conditional_edges("search", router, {"fetch":"fetch", "write":"write"})
g.add_conditional_edges("fetch", router, {"fetch":"fetch", "write":"write"})
g.add_edge("write", END)
agent = g.compile()
print("Agent graph đã compile ✅")

Agent graph đã compile ✅


**Vòng lặp:** entry → `search` → router quyết định (`fetch` nếu thiếu nguồn / `write` nếu đủ) →
`fetch` lại qua router → ... → `write` → END. Router là **bộ não điều phối**; trong production nó là LLM
quyết định bước kế dựa trên ngữ cảnh, ở đây mình dùng rule-based để chạy thật không cần LLM.

## Câu 7 — Guardrail giới hạn số bước — chạy thật ✅

In [5]:
# Guardrail nằm trong router: state["step"] >= max_steps -> ép sang "write"
# Test với max_steps khác nhau
print("=== Chạy với max_steps=5 (bình thường) ===")
r1 = agent.invoke({"topic":"AI trong ngân hàng VN","findings":[],"step":0,"max_steps":5,"report":""})
print(f"-> Dừng sau {r1['step']} bước\n")

print("=== Chạy với max_steps=1 (guardrail siết chặt) ===")
r2 = agent.invoke({"topic":"AI trong ngân hàng VN","findings":[],"step":0,"max_steps":1,"report":""})
print(f"-> Dừng sau {r2['step']} bước (guardrail ép viết sớm)")

=== Chạy với max_steps=5 (bình thường) ===
  [Bước 1] 🔍 web_search
  [Bước 2] 📄 fetch_page
  [Bước 3] ✍️ write_section
-> Dừng sau 3 bước

=== Chạy với max_steps=1 (guardrail siết chặt) ===
  [Bước 1] 🔍 web_search
  [Bước 2] ✍️ write_section
-> Dừng sau 2 bước (guardrail ép viết sớm)


**Vì sao cần guardrail giới hạn bước?**
Agent dùng LLM quyết định bước kế → có thể **lặp vô hạn**: tìm mãi không thấy đủ, gọi tool liên tục, hoặc
hai node đẩy qua đẩy lại. Mỗi bước = 1 lần gọi LLM = **tốn tiền + thời gian**, loop vô hạn có thể "đốt"
hết ngân sách API trong vài phút. `max_steps` là **cầu dao an toàn**: chạm giới hạn → ép agent kết thúc
(viết báo cáo với những gì đã có). Đây là guardrail TỐI THIỂU bắt buộc cho mọi agent production.

## Câu 8 — Observability (trace)

In [6]:
# Production: LangSmith/LangFuse tự log mọi bước. Ở đây ta tự log trace.
def run_with_trace(topic, max_steps=5):
    trace = []
    state = {"topic":topic,"findings":[],"step":0,"max_steps":max_steps,"report":""}
    print(f"🔬 TRACE cho chủ đề: '{topic}'\n" + "="*50)
    for event in agent.stream(state):
        for node, output in event.items():
            entry = {"node": node, "step": output.get("step"),
                     "new_findings": len(output.get("findings",[]))}
            trace.append(entry)
            print(f"  ▸ Node '{node}' | step={entry['step']} | +{entry['new_findings']} finding")
    print("="*50)
    return trace

trace = run_with_trace("AI trong ngân hàng VN")
print(f"\nTrace gồm {len(trace)} bước — mỗi bước ghi: node nào, step mấy, thêm mấy finding")

🔬 TRACE cho chủ đề: 'AI trong ngân hàng VN'
  [Bước 1] 🔍 web_search
  ▸ Node 'search' | step=1 | +1 finding
  [Bước 2] 📄 fetch_page
  ▸ Node 'fetch' | step=2 | +1 finding
  [Bước 3] ✍️ write_section
  ▸ Node 'write' | step=3 | +0 finding

Trace gồm 3 bước — mỗi bước ghi: node nào, step mấy, thêm mấy finding


**Observability** = khả năng "nhìn vào trong" agent khi chạy. **LangSmith/LangFuse** (production) tự động
log: mỗi bước gọi node/tool gì, input/output, token dùng, thời gian, lỗi. **Vì sao cần?** Agent là hộp đen
nhiều bước — khi nó hành xử sai, không có trace thì **không thể debug**. Trace ở trên ghi lại luồng thật:
search → fetch → write, mỗi bước thêm finding. Đính kèm trace này khi nộp bài.

## Câu 9 — Chạy agent với 1 chủ đề — chạy thật ✅

In [7]:
result = agent.invoke({"topic":"Ứng dụng AI trong ngân hàng Việt Nam",
                       "findings":[],"step":0,"max_steps":5,"report":""})
print(result["report"])

  [Bước 1] 🔍 web_search
  [Bước 2] 📄 fetch_page
  [Bước 3] ✍️ write_section
# Báo cáo: Ứng dụng AI trong ngân hàng Việt Nam

[Nguồn 1] Ứng dụng AI trong ngân hàng Việt Nam: lĩnh vực đang tăng trưởng mạnh, nhiều ngân hàng VN đã triển khai.
[Nguồn 2] Báo cáo 2025: 60% ngân hàng VN dùng AI cho CSKH và phát hiện gian lận.

*(Tổng hợp từ 2 nguồn)*


**Báo cáo đầu ra** (chạy thật ở trên) có **trích dẫn nguồn** ([Nguồn 1], [Nguồn 2]) và ghi rõ số nguồn
tổng hợp. Trong production với LLM + web_search thật, báo cáo sẽ dài và chi tiết hơn, nhưng **cấu trúc
agent giữ nguyên**: thu thập → tích lũy → tổng hợp có trích dẫn.

## Câu 10 — Phân tích: agent có đi sai/lặp không?

**Quan sát từ trace:**
- Với `max_steps=5`: agent chạy gọn search → fetch → write (3 bước), KHÔNG lặp thừa. ✅
- Router dừng đúng lúc khi đủ 2 nguồn.

**Các lỗi agent THƯỜNG gặp (và cách xử lý):**
1. **Lặp vô hạn** (tìm mãi không dừng) → **guardrail max_steps** (câu 7) ép dừng. ✅ đã có.
2. **Gọi sai tool** (chọn fetch khi nên search) → cải thiện **mô tả tool** (câu 4) + few-shot ví dụ.
3. **Bỏ qua memory** (tổng hợp thiếu nguồn đã tìm) → đảm bảo state tích lũy đúng (`operator.add`). ✅
4. **Đi lạc chủ đề** (tool trả nội dung không liên quan) → thêm bước **validate/lọc** findings trước khi write.

**Cách phát hiện:** chính là nhờ **observability** (câu 8) — đọc trace thấy ngay bước nào lặp/sai.
Không có trace thì không sửa được.

## Câu 11 — 2 rủi ro production & giảm thiểu

**1. Chi phí & loop không kiểm soát (runaway cost).**
Agent tự quyết số bước → một query xấu có thể kích hoạt hàng trăm lần gọi LLM/tool, "đốt" ngân sách API
và gây độ trễ lớn. Đặc biệt nguy hiểm khi nhiều user chạy đồng thời.
> **Giảm thiểu:** guardrail nhiều tầng — giới hạn số bước (max_steps), giới hạn token/thời gian mỗi lần chạy,
> đặt **budget cap** theo user/ngày, cảnh báo khi vượt ngưỡng. Cache kết quả tool lặp lại.

**2. Hallucination & nguồn không đáng tin (mất kiểm soát chất lượng).**
Agent có thể bịa thông tin, trích dẫn nguồn sai/không tồn tại, hoặc dùng nguồn kém chất lượng từ web —
nguy hiểm khi báo cáo dùng cho quyết định kinh doanh.
> **Giảm thiểu:** bắt agent **chỉ trích dẫn nguồn thật đã fetch** (như RAG — A12), thêm bước **verify**
> đối chiếu claim với nguồn, whitelist domain uy tín, và **human-in-the-loop** duyệt báo cáo quan trọng
> trước khi dùng. Hiển thị nguồn rõ ràng để người đọc tự kiểm chứng.

> Rủi ro khác: bảo mật (agent gọi tool truy cập dữ liệu nhạy cảm — cần phân quyền tool), prompt injection
> (nội dung web độc hại điều khiển agent — cần lọc/sandbox tool output), và độ tin cậy (agent fail giữa
> chừng — cần retry + fallback).